# ITSRS — Results Extraction Notebook

Runs your trained YOLOv11s detector, EfficientNet-B0 classifier, and EasyOCR
reader against your real test sets and logs every metric for the paper's
Results section (Tables 1.3–1.10) into `results/results_log.json`.

Wired to your actual project structure at
`D:\WorkSpace\Traffic Sign Recognition System`. Run top-to-bottom.


## 0. Setup & Config

In [ ]:
# Install once if needed (uncomment)
# !pip install ultralytics easyocr scikit-learn pandas torch torchvision opencv-python --quiet

import os
import json
import time
from pathlib import Path
from datetime import datetime

import cv2
import torch
import torch.nn as nn
import pandas as pd
from ultralytics import YOLO


In [ ]:
PROJECT_ROOT = Path(r"D:\WorkSpace\Traffic Sign Recognition System")

YOLO_WEIGHTS = PROJECT_ROOT / r"runs\detect\runs\detect\indian_yolo11s_production_v1\weights\best.pt"
GTSRB_DATA_YAML = PROJECT_ROOT / r"configs\gtsrb.yaml"
INDIAN_DATA_YAML = PROJECT_ROOT / r"data\Indian_Traffic_Signs_Unified\data.yaml"

EFFICIENTNET_WEIGHTS = PROJECT_ROOT / r"models\classification\efficientnet_b0_best.pth"
CLASS_TEST_DIR = PROJECT_ROOT / r"data\raw\indian\indian_85_class\test"

OCR_TEST_CSV = PROJECT_ROOT / r"configs\ocr_test_labels.csv"
OCR_TEST_DIR = PROJECT_ROOT / r"test_images"

FPS_TEST_VIDEO_DIR = PROJECT_ROOT / r"test_videos"

RESULTS_LOG_PATH = PROJECT_ROOT / r"results\results_log.json"
RESULTS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Sanity check every path exists before running anything else
for name, p in [
    ("YOLO_WEIGHTS", YOLO_WEIGHTS), ("GTSRB_DATA_YAML", GTSRB_DATA_YAML),
    ("INDIAN_DATA_YAML", INDIAN_DATA_YAML), ("EFFICIENTNET_WEIGHTS", EFFICIENTNET_WEIGHTS),
    ("CLASS_TEST_DIR", CLASS_TEST_DIR), ("OCR_TEST_CSV", OCR_TEST_CSV),
    ("OCR_TEST_DIR", OCR_TEST_DIR), ("FPS_TEST_VIDEO_DIR", FPS_TEST_VIDEO_DIR),
]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {name}: {p}")


## 1. Detection metrics — YOLOv11s (Table 1.8)

Validates on GTSRB and the unified Indian test set separately.

In [ ]:
def run_yolo_val(weights_path, data_yaml, split="test"):
    model = YOLO(str(weights_path))
    metrics = model.val(data=str(data_yaml), split=split, verbose=False)
    return {
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    }

detection_results = {}

print("Validating YOLOv11s on GTSRB...")
detection_results["gtsrb"] = run_yolo_val(YOLO_WEIGHTS, GTSRB_DATA_YAML)
print(detection_results["gtsrb"])

print("Validating YOLOv11s on unified Indian test set...")
detection_results["indian_test"] = run_yolo_val(YOLO_WEIGHTS, INDIAN_DATA_YAML)
print(detection_results["indian_test"])


## 2. Classification metrics — EfficientNet-B0 (Table 1.5 / Section 1.5.1)

`.pth` files can hold either a full pickled model or just a `state_dict`.
This cell auto-detects which one you have and loads it correctly either way
— no need to know in advance which format your training script used.

In [ ]:
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = datasets.ImageFolder(str(CLASS_TEST_DIR), transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
class_names = test_dataset.classes
num_classes = len(class_names)
print(f"Found {num_classes} classes, {len(test_dataset)} test images")

def load_efficientnet(weights_path, num_classes, device):
    checkpoint = torch.load(str(weights_path), map_location=device)

    # Case 1: checkpoint is already a full nn.Module
    if isinstance(checkpoint, nn.Module):
        model = checkpoint

    # Case 2: checkpoint is a state_dict (plain dict of tensors, possibly nested under 'model'/'state_dict')
    else:
        state_dict = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
        state_dict = checkpoint.get("model", state_dict) if isinstance(checkpoint, dict) else state_dict

        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

        # strip a possible 'module.' prefix from DataParallel-saved checkpoints
        cleaned = { (k[7:] if k.startswith("module.") else k): v for k, v in state_dict.items() }
        model.load_state_dict(cleaned)

    model.to(device)
    model.eval()
    return model

clf_model = load_efficientnet(EFFICIENTNET_WEIGHTS, num_classes, DEVICE)
print("EfficientNet-B0 loaded.")


In [ ]:
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = clf_model(images.to(DEVICE))
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro")

classification_results = {
    "accuracy": float(acc),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(f1),
}
print(classification_results)

report_text = classification_report(all_labels, all_preds, target_names=class_names)
print(report_text)

report_path = PROJECT_ROOT / "results" / "classification_report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report_text)
print("Full per-class report saved to", report_path)


## 3. OCR accuracy — EasyOCR (Table 1.9)

Reads `configs\ocr_test_labels.csv` and resolves each row against
`test_images\`. Expected CSV columns: `file_name` (relative to
`OCR_TEST_DIR`), `ground_truth`, `condition` (daytime / low_res / night).
If your column names differ, adjust the `.rename()` call below.

In [ ]:
import easyocr

reader = easyocr.Reader(["en"], gpu=(DEVICE == "cuda"))

ocr_df = pd.read_csv(OCR_TEST_CSV)
ocr_df.columns = [c.strip().lower() for c in ocr_df.columns]
# Normalize likely column-name variants to a consistent schema
rename_map = {
    "filename": "file_name", "file": "file_name", "image": "file_name",
    "label": "ground_truth", "gt": "ground_truth", "text": "ground_truth",
}
ocr_df = ocr_df.rename(columns={k: v for k, v in rename_map.items() if k in ocr_df.columns})

ocr_df["full_path"] = ocr_df["file_name"].apply(lambda f: str(OCR_TEST_DIR / f))

def ocr_predict(path):
    if not Path(path).exists():
        return None
    result = reader.readtext(path)
    return result[0][1].strip() if result else ""

ocr_df["prediction"] = ocr_df["full_path"].apply(ocr_predict)
ocr_df["correct"] = ocr_df["prediction"].astype(str).str.strip() == ocr_df["ground_truth"].astype(str).str.strip()

ocr_results = {"overall": float(ocr_df["correct"].mean())}

if "condition" in ocr_df.columns:
    per_condition = ocr_df.groupby("condition")["correct"].mean().to_dict()
    ocr_results.update({f"condition_{k}": float(v) for k, v in per_condition.items()})

print(ocr_results)

detailed_path = PROJECT_ROOT / "results" / "ocr_eval_detailed.csv"
detailed_path.parent.mkdir(parents=True, exist_ok=True)
ocr_df.to_csv(detailed_path, index=False)
print("Detailed OCR predictions saved to", detailed_path)


## 4. End-to-end FPS benchmark (Section 1.5.2)

`test_videos\` holds video files, not loose frames — this cell pulls up to
500 frames from the first video it finds there, then times YOLO-only,
YOLO+EfficientNet, and the full pipeline over the same frame set.

In [ ]:
def extract_frames(video_dir, max_frames=500):
    video_files = list(Path(video_dir).glob("*.mp4")) + list(Path(video_dir).glob("*.avi")) + list(Path(video_dir).glob("*.mov"))
    if not video_files:
        raise FileNotFoundError(f"No video files found in {video_dir}")
    cap = cv2.VideoCapture(str(video_files[0]))
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    print(f"Extracted {len(frames)} frames from {video_files[0].name}")
    return frames

frames = extract_frames(FPS_TEST_VIDEO_DIR, max_frames=500)

detector = YOLO(str(YOLO_WEIGHTS))

def time_stage(fn, frames, label):
    start = time.time()
    for f in frames:
        fn(f)
    elapsed = time.time() - start
    fps = len(frames) / elapsed if elapsed > 0 else 0
    print(f"{label}: {fps:.1f} FPS ({elapsed:.2f}s total, {len(frames)} frames)")
    return fps

# Stage A: YOLO only
fps_yolo_only = time_stage(
    lambda f: detector.predict(f, verbose=False),
    frames, "YOLO only"
)

# Stage B: YOLO + EfficientNet
def yolo_plus_classifier(frame):
    results = detector.predict(frame, verbose=False)
    for r in results:
        for box in r.boxes.xyxy.tolist():
            x1, y1, x2, y2 = map(int, box)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            tensor = transform(torch.from_numpy(crop).permute(2, 0, 1).float() / 255).unsqueeze(0)
            with torch.no_grad():
                clf_model(tensor.to(DEVICE))

fps_yolo_clf = time_stage(yolo_plus_classifier, frames, "YOLO + EfficientNet")

# Stage C: YOLO + EfficientNet + OCR
def full_pipeline(frame):
    results = detector.predict(frame, verbose=False)
    for r in results:
        for box in r.boxes.xyxy.tolist():
            x1, y1, x2, y2 = map(int, box)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            tensor = transform(torch.from_numpy(crop).permute(2, 0, 1).float() / 255).unsqueeze(0)
            with torch.no_grad():
                clf_model(tensor.to(DEVICE))
            reader.readtext(crop)

fps_full_pipeline = time_stage(full_pipeline, frames, "Full pipeline (+ OCR)")

fps_results = {
    "yolo_only": fps_yolo_only,
    "yolo_plus_efficientnet": fps_yolo_clf,
    "full_pipeline_with_ocr": fps_full_pipeline,
}


## 5. Save everything to `results\results_log.json`

In [ ]:
results_log_entry = {
    "timestamp": str(datetime.now()),
    "detection": detection_results,
    "classification": classification_results,
    "ocr": ocr_results,
    "fps": fps_results,
}

with open(RESULTS_LOG_PATH, "a") as f:
    f.write(json.dumps(results_log_entry) + "\n")

print("Saved results to", RESULTS_LOG_PATH)
print(json.dumps(results_log_entry, indent=2))


## 6. Summary table (matches your paper's Table layout)

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "mAP@0.5 (GTSRB)", "mAP@0.5:0.95 (GTSRB)", "Precision (GTSRB)", "Recall (GTSRB)",
        "mAP@0.5 (Indian)", "mAP@0.5:0.95 (Indian)", "Precision (Indian)", "Recall (Indian)",
        "Classification accuracy", "Macro F1",
        "OCR overall accuracy",
        "FPS - YOLO only", "FPS - + EfficientNet", "FPS - + OCR",
    ],
    "Value": [
        detection_results["gtsrb"]["mAP50"], detection_results["gtsrb"]["mAP50_95"],
        detection_results["gtsrb"]["precision"], detection_results["gtsrb"]["recall"],
        detection_results["indian_test"]["mAP50"], detection_results["indian_test"]["mAP50_95"],
        detection_results["indian_test"]["precision"], detection_results["indian_test"]["recall"],
        classification_results["accuracy"], classification_results["macro_f1"],
        ocr_results["overall"],
        fps_results["yolo_only"], fps_results["yolo_plus_efficientnet"], fps_results["full_pipeline_with_ocr"],
    ],
})
summary
